# OPRB — PaddleOCR PP-OCRv5 Mobile English · v2: môi trường riêng

Notebook tự tạo **virtual environment không dùng system site-packages** trong `/tmp`, rồi chạy toàn bộ benchmark bằng Python riêng. ModelScope không nhìn thấy PyTorch của Kaggle; Paddle dùng bộ thư viện CUDA cài trong môi trường của mình. Không cài/gỡ PyTorch hoặc đổi NCCL trong kernel Kaggle.

**Cách chạy:** thay bản cũ bằng notebook này, bật **Internet + GPU**, restart kernel một lần để dọn trạng thái cũ rồi **Run All**. Không cần chọn kernel khác. Mục 1 cài runtime một lần mỗi phiên; các lần chạy lại tái dùng runtime đó. SDK được kiểm tra trước khi tải dữ liệu.

- Model: `PP-OCRv5_mobile_det` + `en_PP-OCRv5_mobile_rec`, trọng số pretrained chính thức, revision cố định. Một pipeline tính là một mô hình, thứ tư sau Florence/EasyOCR/docTR; còn Tesseract theo kế hoạch.
- Giữ nguyên suy luận: cạnh dài detection tối đa 960; batch crop GPU 16/CPU 4; tắt xoay ảnh, nắn trang, TensorRT và HPI. Không fine-tune hoặc chỉnh ngưỡng theo test.
- Mặc định thử cùng 10 ảnh, seed 42. Chạy ổn thì đổi `MAX_SAMPLES_PER_CATEGORY = None` ở Mục 2 và chạy lại Mục 2–4 để chọn đủ 9.000 ảnh.
- Chỉ ảnh che khuất được đưa vào model. GT gốc chỉ dùng để chấm điểm và được lưu nguyên trong CSV.
- Điểm chính bỏ đúng hai token cấu trúc `##LTLine##`, `##LTFigure##`. `raw_*`/`raw_metrics` giữ chính sách cũ để đối chiếu CSV cũ. Chỉ so sánh chỉ số cùng chính sách giữa các model.
- Ghi checkpoint sau mỗi trang, log `PROGRESS` khoảng mỗi phút, giới hạn đánh giá 10 giờ để kịp xuất kết quả. Saved Version mới: gắn output cũ làm Input rồi điền `RESUME_FROM`.

Nguồn: [Paddle/PyTorch có phụ thuộc CUDA khác nhau](https://github.com/PaddlePaddle/Paddle/issues/75926), [ModelScope chỉ import Torch khi tìm thấy gói](https://github.com/modelscope/modelscope/blob/master/modelscope/utils/logger.py), [PaddleOCR 3.4.1](https://github.com/PaddlePaddle/PaddleOCR/tree/v3.4.1), [detector](https://huggingface.co/PaddlePaddle/PP-OCRv5_mobile_det), [recognizer tiếng Anh](https://huggingface.co/PaddlePaddle/en_PP-OCRv5_mobile_rec), [OPRB](https://huggingface.co/datasets/kpurkayastha/OPRB).

## 1. Cài runtime Paddle riêng

Chạy cell này một lần mỗi phiên Kaggle. Môi trường nằm trong `/tmp`, không được gói vào kết quả. [pip --python](https://pip.pypa.io/en/stable/topics/python-option/) cài đúng vào venv; không thay thư viện của kernel. CPU được chọn khi không có GPU hoặc `INSTALL_DEVICE='cpu'`.

In [ ]:
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import venv

INSTALL_DEVICE = 'auto'  # 'auto', 'gpu', hoặc 'cpu'.
assert INSTALL_DEVICE in {'auto', 'gpu', 'cpu'}
if not (3, 9) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError('Runtime Paddle 3.2.0 cần Python 3.9–3.13; kiểm tra môi trường Kaggle.')
pip_version = tuple(int(part) for part in importlib.metadata.version('pip').split('.')[:2])
if pip_version < (22, 3):
    raise RuntimeError('Cần pip >=22.3 để cài vào Python riêng bằng --python.')

NVIDIA_INFO = []
if shutil.which('nvidia-smi'):
    try:
        probe = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=15, check=False,
        )
        if probe.returncode == 0:
            NVIDIA_INFO = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
    except (OSError, subprocess.TimeoutExpired) as exc:
        print('GPU probe:', exc, flush=True)
if INSTALL_DEVICE == 'gpu' and not NVIDIA_INFO:
    raise RuntimeError('Chưa thấy GPU NVIDIA; bật GPU của Kaggle hoặc chọn cpu.')
INSTALL_GPU = bool(NVIDIA_INFO) and INSTALL_DEVICE != 'cpu'
if INSTALL_GPU:
    driver = tuple(int(part) for part in NVIDIA_INFO[0].rsplit(',', 1)[1].strip().split('.'))
    if driver < (450, 80, 2):
        raise RuntimeError('Driver quá cũ cho wheel Paddle đã chọn.')
    wheel_variant = 'cu126' if driver >= (550, 54, 14) else 'cu118'
else:
    wheel_variant = 'cpu'
runtime_name = f'oprb_paddle_py{sys.version_info.major}{sys.version_info.minor}_{wheel_variant}_v2'
PADDLE_ENV = Path(tempfile.gettempdir()) / runtime_name
PADDLE_PYTHON = PADDLE_ENV / 'bin' / 'python'
if not PADDLE_PYTHON.is_file():
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(PADDLE_ENV)
if 'include-system-site-packages = false' not in (PADDLE_ENV / 'pyvenv.cfg').read_text().lower():
    raise RuntimeError('Runtime phải tách khỏi system site-packages; chọn thư mục runtime mới.')

# pip --python hỗ trợ môi trường không có pip: không đổi gói trong kernel Kaggle.
PIP = [sys.executable, '-m', 'pip', '--python', str(PADDLE_PYTHON)]
package = 'paddlepaddle-gpu' if INSTALL_GPU else 'paddlepaddle'
print('Isolated runtime:', PADDLE_ENV, flush=True)
print('Wheel:', package, '3.2.0', wheel_variant, '; GPU:', NVIDIA_INFO or 'CPU', flush=True)
subprocess.check_call(PIP + [
    'install', '-q', '--no-cache-dir', f'{package}==3.2.0',
    '--index-url', f'https://www.paddlepaddle.org.cn/packages/stable/{wheel_variant}/',
])
subprocess.check_call(PIP + [
    'install', '-q', '--no-cache-dir', 'paddleocr==3.4.1', 'paddlex[ocr-core]==3.4.0',
    'huggingface_hub>=0.34,<1', 'jiwer==4.0.0', 'numpy==2.1.3', 'pandas>=2,<3',
    'Pillow>=10,<13', 'matplotlib>=3.8,<4', 'tqdm', 'PyYAML==6.0.2',
])
subprocess.check_call([
    str(PADDLE_PYTHON), '-I', '-c',
    "import importlib.util; assert importlib.util.find_spec('torch') is None; print('Isolation OK: no PyTorch')",
])
print('Cài xong. Tiếp tục Mục 2; không cần đổi kernel.', flush=True)

## 2. Cấu hình

Giữ 5 ảnh mỗi nhóm để thử. Sau khi kiểm tra, đổi `MAX_SAMPLES_PER_CATEGORY=None` và chạy lại Mục 2–4. Nếu toàn bộ test chưa có đủ 6.000/3.000 cặp ảnh/GT, chương trình báo lỗi thay vì coi tập thiếu là đủ 9.000 ảnh.

Saved Version mới: gắn output trước làm Input và điền `RESUME_FROM` vào thư mục chứa CSV/JSON checkpoint. Không trộn checkpoint khác thiết bị, model hoặc chính sách điểm. `DOWNLOAD_DATA=False` hỗ trợ dữ liệu đã có ở `DATA_ROOT`; vẫn phải dùng đúng OPRB test cho bài nộp.

In [ ]:
from pathlib import Path
import json

MAX_SAMPLES_PER_CATEGORY = None  # None để chọn toàn bộ 9.000 ảnh.
DEVICE_PREFERENCE = 'auto'  # 'auto', 'gpu', hoặc 'cpu'.
RESUME_FROM = None  # Ví dụ Path('/kaggle/input/.../oprb_paddleocr_v5_en_gpu_results')
DATA_ROOT = Path('/kaggle/working/oprb_test')
MODEL_DIR = Path('/kaggle/working/paddleocr_models')
CATEGORIES = ['text', 'nontext']
RANDOM_SEED = 42
MAX_EVAL_HOURS = 10.0
CPU_THREADS = 2
DOWNLOAD_DATA = True  # False khi DATA_ROOT đã có sẵn ảnh img/*.jpg và GT gt/*.txt.

assert DEVICE_PREFERENCE in {'auto', 'gpu', 'cpu'}
assert CATEGORIES and set(CATEGORIES) <= {'text', 'nontext'} and len(CATEGORIES) == len(set(CATEGORIES))
assert MAX_SAMPLES_PER_CATEGORY is None or MAX_SAMPLES_PER_CATEGORY > 0
assert MAX_EVAL_HOURS is None or MAX_EVAL_HOURS > 0
if DEVICE_PREFERENCE == 'gpu' and not INSTALL_GPU:
    raise RuntimeError('Chạy lại Mục 1 với INSTALL_DEVICE=gpu trong phiên có GPU.')
resolved_device = ('gpu' if INSTALL_GPU else 'cpu') if DEVICE_PREFERENCE == 'auto' else DEVICE_PREFERENCE
OUTPUT_DIR = Path(f'/kaggle/working/oprb_paddleocr_v5_en_{resolved_device}_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_SETTINGS = {
    'repo_id': 'kpurkayastha/OPRB',
    'data_root': str(DATA_ROOT), 'model_dir': str(MODEL_DIR), 'output_dir': str(OUTPUT_DIR),
    'device_preference': resolved_device,
    'categories': CATEGORIES, 'max_samples_per_category': MAX_SAMPLES_PER_CATEGORY,
    'random_seed': RANDOM_SEED, 'normalize_case': True, 'max_eval_hours': MAX_EVAL_HOURS,
    'cpu_threads': CPU_THREADS, 'download_data': DOWNLOAD_DATA,
    'resume_from': str(RESUME_FROM) if RESUME_FROM is not None else None,
    'sdk_cache_dir': '/tmp/oprb_paddle_cache',
}
print('Device:', resolved_device, '; samples/category:', MAX_SAMPLES_PER_CATEGORY, flush=True)
print('Output:', OUTPUT_DIR, flush=True)

## 3. Chạy benchmark và theo dõi tiến độ

Cell đầu lưu chương trình benchmark và cấu hình vào thư mục output; notebook tự chứa toàn bộ mã, không cần upload thêm `.py`. Cell tiếp theo chạy nó bằng Python riêng và chuyển log lên notebook/Kaggle View Logs. Model chỉ nạp một lần cho cả lượt chạy.

Checkpoint ghi nguyên tử sau mỗi ảnh. Tối đa 10 giờ đánh giá rồi xuất kết quả một phần nếu chưa xong. Dừng bằng interrupt giữ các hàng đã ghi xong. CSV, JSON, biểu đồ và ZIP nằm trong `OUTPUT_DIR`; runtime `/tmp` không nằm trong ZIP.

In [ ]:
import hashlib

BENCHMARK_SOURCE = 'from __future__ import annotations\nimport argparse\nimport importlib.metadata\nimport importlib.util\nimport json\nimport os\nfrom pathlib import Path\nimport sys\n\nparser = argparse.ArgumentParser(description=\'OPRB PaddleOCR benchmark in an isolated environment\')\nparser.add_argument(\'--config\', required=True)\nargs = parser.parse_args()\nSETTINGS = json.loads(Path(args.config).read_text(encoding=\'utf-8\'))\nif sys.prefix == sys.base_prefix:\n    raise RuntimeError(\'Run this script with the isolated virtual-environment Python.\')\nif importlib.util.find_spec(\'torch\') is not None:\n    raise RuntimeError(\'The isolated Paddle environment must not contain PyTorch or inherit system site-packages.\')\nos.environ[\'PADDLE_PDX_EAGER_INIT\'] = \'False\'\nos.environ[\'PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK\'] = \'True\'\nos.environ[\'PADDLE_PDX_DISABLE_DEVICE_FALLBACK\'] = \'True\'\nos.environ[\'PADDLE_PDX_CACHE_HOME\'] = SETTINGS.get(\'sdk_cache_dir\', \'/tmp/oprb_paddle_cache\')\nos.environ[\'MPLBACKEND\'] = \'Agg\'\nos.environ.setdefault(\'MPLCONFIGDIR\', \'/tmp/oprb_paddle_matplotlib\')\nprint(\'ISOLATED RUNTIME:\', sys.executable, \'; Python\', sys.version.split()[0], \'; PyTorch absent\', flush=True)\n\n\nprint(\'Import Paddle and validate SDK\', flush=True)\n\nimport gc, hashlib, json, os, platform, random, re, shutil, tarfile, time, unicodedata\nfrom collections import Counter\nfrom importlib.metadata import version\nfrom pathlib import Path\n\n# Chỉ tắt kiểm tra kết nối tới các kho khác; trọng số tải rõ ràng từ Hugging Face ở Mục 7.\nos.environ[\'PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK\'] = \'True\'\nos.environ[\'PADDLE_PDX_DISABLE_DEVICE_FALLBACK\'] = \'True\'\n# Chỉ dùng inference, không khởi tạo sớm các repository training của PaddleX.\n# Phải đặt trước lần import paddlex/paddleocr đầu tiên trong kernel.\nos.environ[\'PADDLE_PDX_EAGER_INIT\'] = \'False\'\n\nimport numpy as np\nimport pandas as pd\nimport cv2\nimport paddle\nimport yaml\nfrom PIL import Image\ndef display(value):\n    if isinstance(value, pd.DataFrame):\n        print(value.to_string(index=False), flush=True)\nfrom huggingface_hub import hf_hub_download\nfrom jiwer import cer, wer\nfrom tqdm import tqdm\nimport matplotlib.pyplot as plt\n\nCUDA_AVAILABLE = bool(paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0)\nprint(\'Python:\', platform.python_version(), flush=True)\nprint(\'Paddle:\', paddle.__version__, \'; CUDA GPU dùng được:\', CUDA_AVAILABLE, flush=True)\nif CUDA_AVAILABLE:\n    print(\'GPU đầu tiên:\', paddle.device.cuda.get_device_name(0), flush=True)\nelse:\n    print(\'Sẽ chạy trên CPU. Muốn dùng GPU: bật GPU và chạy lại Mục 1 trong kernel mới.\', flush=True)\n\n\ndef load_paddleocr_class():\n    """Kiểm tra SDK sớm; không sửa/xóa trạng thái nội bộ PaddleX trong kernel lỗi."""\n    for package, expected in [(\'paddleocr\', \'3.4.1\'), (\'paddlex\', \'3.4.0\')]:\n        if version(package) != expected:\n            raise RuntimeError(f\'Cần {package}=={expected}; chạy Mục 1 và restart kernel nếu đã import bản khác.\')\n    os.environ[\'PADDLE_PDX_EAGER_INIT\'] = \'False\'\n    try:\n        from paddleocr import PaddleOCR\n    except RuntimeError as exc:\n        if \'PDX has already been initialized\' in str(exc):\n            raise RuntimeError(\n                \'PaddleX còn trạng thái khởi tạo trong kernel cũ. Restart kernel, \'\n                \'rồi chạy từ Mục 2; không chỉ chạy lại cell lỗi. Không cần xóa dữ liệu/checkpoint.\'\n            ) from exc\n        raise\n    return PaddleOCR\n\n\nPaddleOCR = load_paddleocr_class()\nprint(\'SDK import OK:\', \'PaddleOCR\', version(\'paddleocr\'), \'; PaddleX\', version(\'paddlex\'), flush=True)\n\n\n\nprint(\'Configuration\', flush=True)\n\nREPO_ID = SETTINGS.get(\'repo_id\', \'kpurkayastha/OPRB\')\nMODEL_ID = \'PaddleOCR:PP-OCRv5_mobile_det+en_PP-OCRv5_mobile_rec\'\nDATA_ROOT = Path(SETTINGS[\'data_root\'])\nMODEL_DIR = Path(SETTINGS[\'model_dir\'])\nDEVICE_PREFERENCE = SETTINGS[\'device_preference\']\nassert DEVICE_PREFERENCE in {\'auto\', \'gpu\', \'cpu\'}\nif DEVICE_PREFERENCE == \'gpu\' and (not CUDA_AVAILABLE):\n    raise RuntimeError(\'Đã chọn gpu nhưng Paddle chưa dùng được GPU. Kiểm tra Mục 1–2.\')\nUSE_CUDA = CUDA_AVAILABLE and DEVICE_PREFERENCE != \'cpu\'\nDEVICE = \'gpu:0\' if USE_CUDA else \'cpu\'\nDEVICE_NAME = paddle.device.cuda.get_device_name(0) if USE_CUDA else \'CPU\'\npaddle.set_device(DEVICE)\nOUTPUT_DIR = Path(SETTINGS[\'output_dir\'])\nRESUME_FROM = Path(SETTINGS[\'resume_from\']) if SETTINGS.get(\'resume_from\') else None\nCATEGORIES = SETTINGS[\'categories\']\nMAX_SAMPLES_PER_CATEGORY = SETTINGS[\'max_samples_per_category\']\nRANDOM_SEED = SETTINGS.get(\'random_seed\', 42)\nNORMALIZE_CASE = SETTINGS.get(\'normalize_case\', True)\nMAX_EVAL_HOURS = SETTINGS.get(\'max_eval_hours\', 10.0)\nCPU_THREADS = SETTINGS.get(\'cpu_threads\', 2)\nGT_LAYOUT_MARKERS = [\'##LTLine##\', \'##LTFigure##\']\nMETRIC_POLICY_ID = \'oprb_macro_nfkc_layout_markers_removed_v1\'\nDETECT_NETWORK = \'PP-OCRv5_mobile_det\'\nRECOG_NETWORK = \'en_PP-OCRv5_mobile_rec\'\nMODEL_SPECS = {\'detector\': {\'name\': DETECT_NETWORK, \'repo_id\': f\'PaddlePaddle/{DETECT_NETWORK}\', \'revision\': \'0d63e78e2b680928f6b1747d76a08db6e645efb7\'}, \'recognizer\': {\'name\': RECOG_NETWORK, \'repo_id\': f\'PaddlePaddle/{RECOG_NETWORK}\', \'revision\': \'267c36e24c331595590fe7bd72bde2436fd286f2\'}}\nMODEL_FILES = [\'inference.json\', \'inference.pdiparams\', \'inference.yml\']\nRECOGNITION_BATCH_SIZE = 16 if USE_CUDA else 4\nOCR_KWARGS = {\'use_doc_orientation_classify\': False, \'use_doc_unwarping\': False, \'use_textline_orientation\': False, \'text_det_limit_side_len\': 960, \'text_det_limit_type\': \'max\', \'text_det_thresh\': 0.3, \'text_det_box_thresh\': 0.6, \'text_det_unclip_ratio\': 1.5, \'text_recognition_batch_size\': RECOGNITION_BATCH_SIZE, \'text_rec_score_thresh\': 0.0, \'return_word_box\': False}\nRUNTIME_KWARGS = {\'device\': DEVICE, \'enable_hpi\': False, \'use_tensorrt\': False, \'precision\': \'fp32\', \'enable_mkldnn\': False, \'enable_cinn\': False, \'cpu_threads\': CPU_THREADS}\nassert MAX_EVAL_HOURS is None or MAX_EVAL_HOURS > 0\nassert MAX_SAMPLES_PER_CATEGORY is None or MAX_SAMPLES_PER_CATEGORY > 0\nassert CPU_THREADS > 0 and RECOGNITION_BATCH_SIZE > 0\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nMODEL_DIR.mkdir(parents=True, exist_ok=True)\npaddle.seed(RANDOM_SEED)\nprint(\'Pipeline:\', MODEL_ID, flush=True)\nprint(\'Thiết bị:\', DEVICE, DEVICE_NAME, flush=True)\nprint(\'Mỗi loại:\', MAX_SAMPLES_PER_CATEGORY, \'; seed:\', RANDOM_SEED, flush=True)\nprint(\'Điểm chính:\', METRIC_POLICY_ID, flush=True)\nprint(\'Output:\', OUTPUT_DIR, flush=True)\n\nruntime_packages = {\n    distribution.metadata[\'Name\']: distribution.version\n    for distribution in importlib.metadata.distributions()\n    if distribution.metadata.get(\'Name\')\n}\n(OUTPUT_DIR / \'runtime_packages.json\').write_text(\n    json.dumps(dict(sorted(runtime_packages.items())), indent=2), encoding=\'utf-8\'\n)\n\n\n\n\nprint(\'Download test images and scoring GT\', flush=True)\n\ndef extract_flat(archive_path: str, destination: Path, allowed_suffix: str) -> int:\n    """Safely copy only files of the specified type; never use archive-supplied paths."""\n    destination.mkdir(parents=True, exist_ok=True)\n    count = 0\n    with tarfile.open(archive_path, mode=\'r:gz\') as archive:\n        for member in tqdm(archive, desc=f\'Extract {destination.parent.name}/{destination.name}\', leave=False):\n            if not member.isfile() or not member.name.lower().endswith(allowed_suffix):\n                continue\n            filename = Path(member.name).name\n            target = destination / filename\n            if not target.exists():\n                stream = archive.extractfile(member)\n                if stream is None:\n                    continue\n                with stream, target.open(\'wb\') as out:\n                    shutil.copyfileobj(stream, out)\n            count += 1\n    return count\n\ndef download_test_category(category: str):\n    if category not in {\'text\', \'nontext\'}:\n        raise ValueError(f\'Unknown category: {category}\')\n    for kind, suffix in ((\'img\', \'.jpg\'), (\'gt\', \'.txt\')):\n        repo_path = f\'test/{category}/{kind}.tar.gz\'\n        target_dir = DATA_ROOT / category / kind\n        existing = list(target_dir.glob(f\'*{suffix}\')) if target_dir.exists() else []\n        expected = 6000 if category == \'text\' else 3000\n        if len(existing) >= expected:\n            print(f\'Reuse {category}/{kind}: {len(existing)} files\')\n            continue\n        print(f\'Downloading {repo_path} ...\')\n        tar_path = hf_hub_download(repo_id=REPO_ID, repo_type=\'dataset\', filename=repo_path)\n        n = extract_flat(tar_path, target_dir, suffix)\n        print(f\'  {category}/{kind}: {n:,} archive members of type {suffix} -> {target_dir}\')\nif SETTINGS.get(\'download_data\', True):\n    for category in CATEGORIES:\n        download_test_category(category)\nelse:\n    print(\'Using local test images/GT from:\', DATA_ROOT, flush=True)\n\n\n\n\nprint(\'Pair test images and GT\', flush=True)\n\ndef parse_occlusion_metadata(stem: str):\n    # OPRB filename convention: OcclusionType__density__batchID... \n    parts = stem.split("__", 2)\n    return {\n        "occlusion_type": parts[0] if parts else "unknown",\n        "occlusion_density": parts[1] if len(parts) > 1 else "unknown",\n    }\n\n\ndef collect_test_pairs(categories):\n    pairs = []\n    for category in categories:\n        image_dir = DATA_ROOT / category / "img"\n        gt_dir = DATA_ROOT / category / "gt"\n        images = {p.stem: p for p in image_dir.glob("*.jpg")}\n        gts = {p.stem: p for p in gt_dir.glob("*.txt")}\n        common = sorted(images.keys() & gts.keys())\n        print(f"{category}: images={len(images):,}, gt={len(gts):,}, paired={len(common):,}")\n        if not common:\n            raise RuntimeError(f"No paired files for {category}. Check extraction and naming.")\n        if MAX_SAMPLES_PER_CATEGORY is not None:\n            rng = random.Random(RANDOM_SEED)\n            common = sorted(rng.sample(common, min(MAX_SAMPLES_PER_CATEGORY, len(common))))\n        for stem in common:\n            pairs.append({\n                "category": category,\n                "sample_id": stem,\n                "image_path": str(images[stem]),\n                "gt_path": str(gts[stem]),\n                **parse_occlusion_metadata(stem),\n            })\n    return pairs\n\n\npairs = collect_test_pairs(CATEGORIES)\nprint(f"Selected {len(pairs):,} samples. Set MAX_SAMPLES_PER_CATEGORY=None to use the entire test set.")\ndisplay(pd.DataFrame(pairs)[["category", "occlusion_type", "occlusion_density", "sample_id"]].head(8))\n\nwith Image.open(pairs[0]["image_path"]) as image:\n    preview = image.copy()\npreview.thumbnail((650, 850))\ndisplay(preview)\nif MAX_SAMPLES_PER_CATEGORY is None:\n    expected_counts = {\'text\': 6000, \'nontext\': 3000}\n    actual_counts = Counter(sample[\'category\'] for sample in pairs)\n    for category in CATEGORIES:\n        if actual_counts[category] != expected_counts[category]:\n            raise RuntimeError(f\'Full test requires {expected_counts[category]} paired {category} pages; got {actual_counts[category]}.\')\npreview.save(OUTPUT_DIR / \'preview.png\')\n\n\n\n\nprint(\'Metrics: layout tokens removed + legacy raw scores\', flush=True)\n\ndef read_gt_tokens(gt_path: str):\n    tokens = []\n    with open(gt_path, \'r\', encoding=\'utf-8-sig\', errors=\'replace\') as file:\n        for line in file:\n            fields = line.rstrip(\'\\r\\n\').split(\'\\t\')\n            if len(fields) >= 10 and fields[0].strip():\n                tokens.append(fields[0].strip())\n    if not tokens:\n        raise ValueError(f\'No GT tokens read from {gt_path}. Check annotation format.\')\n    return tokens\n\n\ndef normalize_text(text: str) -> str:\n    text = unicodedata.normalize(\'NFKC\', text)\n    text = re.sub(r\'\\s+\', \' \', text).strip()\n    return text.casefold() if NORMALIZE_CASE else text\n\n\ndef clean_text(text: str) -> str:\n    normalized = normalize_text(text)\n    markers = {normalize_text(marker) for marker in GT_LAYOUT_MARKERS}\n    return \' \'.join(token for token in normalized.split() if token not in markers)\n\n\ndef bag_of_words_prf(reference: str, prediction: str):\n    ref, hyp = Counter(reference.split()), Counter(prediction.split())\n    matched = sum((ref & hyp).values())\n    ref_n, hyp_n = sum(ref.values()), sum(hyp.values())\n    precision = matched / hyp_n if hyp_n else (1.0 if ref_n == 0 else 0.0)\n    recall = matched / ref_n if ref_n else (1.0 if hyp_n == 0 else 0.0)\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return precision, recall, f1\n\n\ndef score_normalized_pair(ref: str, hyp: str):\n    if not ref:\n        raise ValueError(\'Empty GT transcript after normalization; cannot compute CER/WER.\')\n    precision, recall, f1 = bag_of_words_prf(ref, hyp)\n    return {\n        \'cer\': float(cer(ref, hyp)), \'wer\': float(wer(ref, hyp)),\n        \'token_precision\': precision, \'token_recall\': recall, \'token_f1\': f1,\n        \'gt_words\': len(ref.split()), \'pred_words\': len(hyp.split()),\n    }\n\n\ndef compute_metrics(ground_truth: str, prediction: str):\n    ref_raw, hyp_raw = normalize_text(ground_truth), normalize_text(prediction)\n    ref, hyp = clean_text(ground_truth), clean_text(prediction)\n    return {\n        **score_normalized_pair(ref, hyp),\n        **{f\'raw_{key}\': value for key, value in score_normalized_pair(ref_raw, hyp_raw).items()},\n        \'n_gt_markers_removed\': len(ref_raw.split()) - len(ref.split()),\n        \'n_pred_markers_removed\': len(hyp_raw.split()) - len(hyp.split()),\n    }\n\n\nassert compute_metrics(\'The cat sat\', \'The cat\')[\'wer\'] == 1/3\nassert compute_metrics(\'Hello ##LTLine## world\', \'hello world\')[\'wer\'] == 0\nassert compute_metrics(\'Hello ##LTLine## world\', \'hello world\')[\'raw_wer\'] == 1/3\nprint(\'Toy check:\', compute_metrics(\'Hello ##LTLine## world\', \'hello world\'), flush=True)\nprint(\'First GT tokens:\', read_gt_tokens(pairs[0][\'gt_path\'])[:35], flush=True)\n\n\n\nprint(\'Load official pretrained detector and recognizer\', flush=True)\n\nPaddleOCR = load_paddleocr_class()\n\nif paddle.__version__ != \'3.2.0\':\n    raise RuntimeError(\'Cần PaddlePaddle 3.2.0 như Mục 1.\')\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\nMODEL_COMPONENTS = []\nMODEL_PATHS = {}\nfor component_name, specification in MODEL_SPECS.items():\n    local_dir = MODEL_DIR / specification[\'name\']\n    local_dir.mkdir(parents=True, exist_ok=True)\n    artifacts = []\n    for filename in MODEL_FILES:\n        cached = Path(hf_hub_download(\n            repo_id=specification[\'repo_id\'], repo_type=\'model\',\n            revision=specification[\'revision\'], filename=filename,\n        ))\n        target = local_dir / filename\n        if not target.is_file() or sha256_file(target) != sha256_file(cached):\n            shutil.copy2(cached, target)\n        artifacts.append({\'filename\': filename, \'sha256\': sha256_file(target), \'bytes\': target.stat().st_size})\n    inference_config = yaml.safe_load((local_dir / \'inference.yml\').read_text(encoding=\'utf-8\'))\n    if inference_config.get(\'Global\', {}).get(\'model_name\') != specification[\'name\']:\n        raise ValueError(f\'Model config không khớp: {local_dir}\')\n    MODEL_PATHS[component_name] = local_dir\n    MODEL_COMPONENTS.append({\n        \'component\': component_name,\n        \'architecture\': specification[\'name\'],\n        \'checkpoint_repo\': specification[\'repo_id\'],\n        \'checkpoint_revision\': specification[\'revision\'],\n        \'artifacts\': artifacts,\n        \'weight_file_bytes\': (local_dir / \'inference.pdiparams\').stat().st_size,\n        \'parameter_count\': None,\n        \'parameter_count_note\': \'Not inferred from serialized inference-file size; report weight MiB.\',\n        \'inference_config\': inference_config,\n    })\n\nocr = PaddleOCR(\n    text_detection_model_name=DETECT_NETWORK,\n    text_detection_model_dir=str(MODEL_PATHS[\'detector\']),\n    text_recognition_model_name=RECOG_NETWORK,\n    text_recognition_model_dir=str(MODEL_PATHS[\'recognizer\']),\n    **OCR_KWARGS,\n    **RUNTIME_KWARGS,\n)\npipeline_config_path = OUTPUT_DIR / \'paddlex_pipeline.yaml\'\nocr.export_paddlex_config_to_yaml(str(pipeline_config_path))\nPIPELINE_CONFIG = yaml.safe_load(pipeline_config_path.read_text(encoding=\'utf-8\'))\nif PIPELINE_CONFIG.get(\'use_doc_preprocessor\') or PIPELINE_CONFIG.get(\'use_textline_orientation\'):\n    raise RuntimeError(\'Không được bật mạng phụ trong thí nghiệm này.\')\nEFFECTIVE_OCR_SETTINGS = json.loads(json.dumps({\n    \'pipeline\': PIPELINE_CONFIG,\n    \'explicit_ocr_options\': OCR_KWARGS,\n    \'runtime\': RUNTIME_KWARGS,\n}))\nprint(\'Đã tải:\', [(c[\'component\'], c[\'architecture\']) for c in MODEL_COMPONENTS], flush=True)\nprint(\'Tổng trọng số (MiB):\', sum(c[\'weight_file_bytes\'] for c in MODEL_COMPONENTS) / 1024**2, flush=True)\nprint(\'Thiết bị:\', DEVICE_NAME, \'; batch crop:\', RECOGNITION_BATCH_SIZE, flush=True)\nprint(\'Config:\', pipeline_config_path, flush=True)\n\n\n\nprint(\'Single-image warm-up\', flush=True)\n\ndef transcribe_page(image_path: str):\n    """Chỉ nhận đường dẫn ảnh che khuất, không nhận GT."""\n    if USE_CUDA:\n        paddle.device.cuda.synchronize(0)\n    started = time.perf_counter()\n    pages = list(ocr.predict(input=str(image_path)))\n    if len(pages) != 1:\n        raise RuntimeError(f\'Expected one page; got {len(pages)}\')\n    page = pages[0]\n    if \'error\' in page:\n        raise RuntimeError(f\'PaddleOCR error: {page["error"]}\')\n    texts = page[\'rec_texts\']\n    if not isinstance(texts, (list, tuple)) or not all(isinstance(text, str) for text in texts):\n        raise TypeError(\'Unexpected PaddleOCR rec_texts schema; expected a list of strings.\')\n    prediction = \' \'.join(text.strip() for text in texts if text.strip())\n    if USE_CUDA:\n        paddle.device.cuda.synchronize(0)\n    elapsed_sec = time.perf_counter() - started\n    return prediction, elapsed_sec, int(len(texts))\n\n\nsample = pairs[0]\nexample_gt = \' \'.join(read_gt_tokens(sample[\'gt_path\']))\nexample_prediction, example_time, example_regions = transcribe_page(sample[\'image_path\'])\nprint(\'CATEGORY:\', sample[\'category\'], flush=True)\nprint(\'IMAGE:\', Path(sample[\'image_path\']).name, flush=True)\nprint(\'LATENCY (s):\', round(example_time, 2), flush=True)\nprint(\'OCR REGIONS:\', example_regions, flush=True)\nprint(\'GT RAW (first 700 chars):\', example_gt[:700], flush=True)\nprint(\'PREDICTION (first 700 chars):\', example_prediction[:700], flush=True)\nprint(\'METRICS:\', compute_metrics(example_gt, example_prediction), flush=True)\n\n\n\nprint(\'Evaluation and checkpoint\', flush=True)\n\nRESULTS_CSV = OUTPUT_DIR / \'per_image_results.csv\'\nCHECKPOINT_CONFIG = OUTPUT_DIR / \'checkpoint_config.json\'\ncheckpoint_settings = {\n    \'dataset\': REPO_ID,\n    \'split\': \'test\',\n    \'runtime_isolation\': \'venv_without_system_site_packages\',\n    \'python_version\': platform.python_version(),\n    \'model\': MODEL_ID,\n    \'detector\': DETECT_NETWORK,\n    \'recognizer\': RECOG_NETWORK,\n    \'components\': MODEL_COMPONENTS,\n    \'quantization\': None,\n    \'precision_option\': \'fp32\',\n    \'metric_policy_id\': METRIC_POLICY_ID,\n    \'excluded_layout_tokens\': GT_LAYOUT_MARKERS,\n    \'ocr_settings\': EFFECTIVE_OCR_SETTINGS,\n    \'reading_order\': \'PaddleOCR rec_texts order; join nonempty region texts with spaces\',\n    \'normalize_case\': NORMALIZE_CASE,\n    \'metric_definition\': \'Macro per-image metrics after layout-token removal; raw_* retains legacy unfiltered scores\',\n    \'device\': DEVICE,\n    \'device_name\': DEVICE_NAME,\n    \'cpu_threads\': CPU_THREADS,\n    \'paddleocr_version\': version(\'paddleocr\'),\n    \'paddlex_version\': version(\'paddlex\'),\n    \'paddle_version\': str(paddle.__version__),\n    \'numpy_version\': np.__version__,\n    \'opencv_version\': cv2.__version__,\n    \'jiwer_version\': version(\'jiwer\'),\n}\n\nif RESUME_FROM is not None and not RESULTS_CSV.exists():\n    resume_root = Path(RESUME_FROM)\n    for filename in (\'checkpoint_config.json\', \'per_image_results.csv\'):\n        if not (resume_root / filename).is_file():\n            raise FileNotFoundError(f\'Missing resume file: {resume_root / filename}\')\n    for filename in (\'checkpoint_config.json\', \'per_image_results.csv\'):\n        source = resume_root / filename\n        target = OUTPUT_DIR / filename\n        if target.exists() and target.read_bytes() != source.read_bytes():\n            raise FileExistsError(f\'Không ghi đè checkpoint khác: {target}\')\n        if source.resolve() != target.resolve():\n            shutil.copy2(source, target)\n    print(\'Restored checkpoint from:\', resume_root, flush=True)\n\nif CHECKPOINT_CONFIG.exists():\n    saved_settings = json.loads(CHECKPOINT_CONFIG.read_text(encoding=\'utf-8\'))\n    if saved_settings != checkpoint_settings:\n        raise ValueError(\'Checkpoint khác mô hình/cấu hình/thiết bị. Chọn OUTPUT_DIR mới cho thí nghiệm khác.\')\nelif RESULTS_CSV.exists():\n    raise ValueError(\'CSV thiếu checkpoint_config.json. Phục hồi cả hai tệp hoặc chọn OUTPUT_DIR mới.\')\nelse:\n    CHECKPOINT_CONFIG.write_text(json.dumps(checkpoint_settings, indent=2), encoding=\'utf-8\')\n\n\ndef sample_key(record):\n    return record[\'category\'], record[\'sample_id\']\n\n\nselected_keys = {sample_key(sample) for sample in pairs}\nif RESULTS_CSV.exists():\n    previous = pd.read_csv(RESULTS_CSV, keep_default_na=False).to_dict(orient=\'records\')\n    if any(record.get(\'model_id\') != MODEL_ID for record in previous):\n        raise ValueError(\'CSV chứa mô hình khác. Chọn OUTPUT_DIR riêng.\')\n    saved_rows = {sample_key(record): record for record in previous}\nelse:\n    saved_rows = {}\nfinished = {key for key, record in saved_rows.items() if record.get(\'status\') == \'ok\'}\npending = [sample for sample in pairs if sample_key(sample) not in finished]\nprint(f\'Reusing {len(finished & selected_keys):,} successful selected pages; {len(pending):,} remaining.\', flush=True)\nevaluation_started = time.perf_counter()\nlast_progress_log = evaluation_started\nsuccessful_count = len(finished & selected_keys)\nerrors_this_run = 0\n\n\ndef format_duration(seconds):\n    hours, remainder = divmod(int(seconds), 3600)\n    minutes, seconds = divmod(remainder, 60)\n    return f\'{hours:02d}:{minutes:02d}:{seconds:02d}\'\n\n\nfor attempted_this_run, sample in enumerate(tqdm(pending, desc=\'PaddleOCR v5 Mobile evaluation\', unit=\'page\', mininterval=60), start=1):\n    if MAX_EVAL_HOURS is not None and time.perf_counter() - evaluation_started >= MAX_EVAL_HOURS * 3600:\n        print(\'Time budget reached. Export this checkpoint, then resume later.\', flush=True)\n        break\n    result = dict(sample)\n    result[\'model_id\'] = MODEL_ID\n    try:\n        ground_truth = \' \'.join(read_gt_tokens(sample[\'gt_path\']))\n        if not clean_text(ground_truth):\n            raise ValueError(\'GT chỉ chứa nhãn cấu trúc; không có văn bản để chấm điểm.\')\n        prediction, elapsed, n_regions = transcribe_page(sample[\'image_path\'])\n        result.update({\n            \'status\': \'ok\',\n            \'prediction\': prediction,\n            \'ground_truth\': ground_truth,\n            \'latency_sec\': elapsed,\n            \'n_text_regions\': n_regions,\n            **compute_metrics(ground_truth, prediction),\n        })\n    except Exception as exc:\n        result.update({\'status\': \'error\', \'error\': repr(exc)})\n        print(\'Failure:\', sample[\'sample_id\'], repr(exc), flush=True)\n        gc.collect()\n        if USE_CUDA:\n            paddle.device.cuda.empty_cache()\n    saved_rows[sample_key(sample)] = result\n    temporary_csv = RESULTS_CSV.with_suffix(\'.csv.tmp\')\n    pd.DataFrame(saved_rows.values()).to_csv(temporary_csv, index=False)\n    temporary_csv.replace(RESULTS_CSV)\n    successful_count += result[\'status\'] == \'ok\'\n    errors_this_run += result[\'status\'] == \'error\'\n    now = time.perf_counter()\n    if attempted_this_run == 1 or now - last_progress_log >= 60 or attempted_this_run == len(pending):\n        elapsed = now - evaluation_started\n        average_sec = elapsed / attempted_this_run\n        unattempted = len(pending) - attempted_this_run\n        print(\n            f\'PROGRESS: {successful_count:,}/{len(pairs):,} successful ({100 * successful_count / len(pairs):.1f}%) | \'\n            f\'attempted this run={attempted_this_run:,}/{len(pending):,} | errors this run={errors_this_run:,} | \'\n            f\'elapsed={format_duration(elapsed)} | avg={average_sec:.2f}s/page | \'\n            f\'ETA for unattempted pages={format_duration(unattempted * average_sec)}\',\n            flush=True,\n        )\n        last_progress_log = now\n\nrows = [record for key, record in saved_rows.items() if key in selected_keys]\nresults = pd.DataFrame(rows)\nif results.empty:\n    raise RuntimeError(\'Chưa có ảnh được thử. Kiểm tra giới hạn thời gian và checkpoint.\')\nprint(\'Saved:\', RESULTS_CSV, flush=True)\nprint(\'Successful:\', (results.status == \'ok\').sum(), \'/\', len(pairs), \'selected pages\', flush=True)\ndisplay(results.reindex(columns=[\n    \'category\', \'occlusion_type\', \'cer\', \'wer\', \'token_f1\',\n    \'latency_sec\', \'n_text_regions\', \'status\', \'error\'\n]).head(10))\n\n\n\nprint(\'Summaries\', flush=True)\n\nok = results.loc[results[\'status\'] == \'ok\'].copy()\nassert len(ok), \'Không có trang chấm điểm được. Kiểm tra cột error.\'\nmetric_cols = [\'cer\', \'wer\', \'token_precision\', \'token_recall\', \'token_f1\', \'latency_sec\']\n# CSV có hàng lỗi có thể khiến pandas đọc các số thành chuỗi khi resume.\nfor column in metric_cols + [\'n_text_regions\']:\n    ok[column] = pd.to_numeric(ok[column], errors=\'raise\')\n\n\ndef aggregate_metrics(df, group_by):\n    return df.groupby(group_by, dropna=False).agg(\n        n=(\'sample_id\', \'count\'),\n        **{f\'mean_{name}\': (name, \'mean\') for name in metric_cols},\n        mean_text_regions=(\'n_text_regions\', \'mean\'),\n    ).reset_index()\n\n\noverall = {f\'mean_{name}\': float(ok[name].mean()) for name in metric_cols}\noverall.update({\n    \'n_selected\': int(len(pairs)),\n    \'n_success\': int(len(ok)),\n    \'n_errors\': int(len(results) - len(ok)),\n    \'n_remaining\': int(len(pairs) - len(ok)),\n    \'evaluation_complete\': bool(len(ok) == len(pairs)),\n    \'n_empty_predictions\': int(ok[\'prediction\'].fillna(\'\').astype(str).str.strip().eq(\'\').sum()),\n    \'mean_text_regions\': float(ok[\'n_text_regions\'].mean()),\n})\nby_category = aggregate_metrics(ok, [\'category\'])\nby_type = aggregate_metrics(ok, [\'category\', \'occlusion_type\'])\nby_density = aggregate_metrics(ok, [\'category\', \'occlusion_density\'])\nprint(\'OVERALL macro-averaged metrics:\', flush=True)\nprint(json.dumps(overall, indent=2), flush=True)\nprint(\'BY OCCLUSION REGION:\', flush=True)\ndisplay(by_category)\nprint(\'BY OCCLUSION TYPE:\', flush=True)\ndisplay(by_type)\nprint(\'BY OCCLUSION DENSITY:\', flush=True)\ndisplay(by_density)\nby_category.to_csv(OUTPUT_DIR / \'summary_by_category.csv\', index=False)\nby_type.to_csv(OUTPUT_DIR / \'summary_by_occlusion_type.csv\', index=False)\nby_density.to_csv(OUTPUT_DIR / \'summary_by_density.csv\', index=False)\n\nregion_times = by_category.set_index(\'category\')[\'mean_latency_sec\']\nif {\'text\', \'nontext\'}.issubset(region_times.index):\n    estimated_full_hours = (6000 * region_times[\'text\'] + 3000 * region_times[\'nontext\']) / 3600\n    print(f\'Ước lượng suy luận 9.000 trang trên {DEVICE_NAME}: {estimated_full_hours:.1f} giờ.\', flush=True)\nprint(\'Số ảnh còn cần chấm thành công:\', overall[\'n_remaining\'], flush=True)\n\n\n# Điểm theo chính sách cũ để đối chiếu các CSV trước khi lọc marker.\nraw_metric_cols = [\'cer\', \'wer\', \'token_precision\', \'token_recall\', \'token_f1\']\nfor name in raw_metric_cols + [\'gt_words\', \'pred_words\']:\n    ok[f\'raw_{name}\'] = pd.to_numeric(ok[f\'raw_{name}\'], errors=\'raise\')\nfor column in [\'n_gt_markers_removed\', \'n_pred_markers_removed\']:\n    ok[column] = pd.to_numeric(ok[column], errors=\'raise\')\nraw_overall = {f\'mean_{name}\': float(ok[f\'raw_{name}\'].mean()) for name in raw_metric_cols}\nraw_overall[\'mean_latency_sec\'] = overall[\'mean_latency_sec\']\noverall[\'n_gt_markers_removed\'] = int(ok[\'n_gt_markers_removed\'].sum())\noverall[\'n_pred_markers_removed\'] = int(ok[\'n_pred_markers_removed\'].sum())\nraw_ok = ok.copy()\nfor name in raw_metric_cols:\n    raw_ok[name] = raw_ok[f\'raw_{name}\']\nfor filename, groups in [\n    (\'raw_summary_by_category.csv\', [\'category\']),\n    (\'raw_summary_by_occlusion_type.csv\', [\'category\', \'occlusion_type\']),\n    (\'raw_summary_by_density.csv\', [\'category\', \'occlusion_density\']),\n]:\n    aggregate_metrics(raw_ok, groups).to_csv(OUTPUT_DIR / filename, index=False)\nprint(\'RAW/LEGACY macro metrics (còn nhãn, để đối chiếu CSV cũ):\', flush=True)\nprint(json.dumps(raw_overall, indent=2), flush=True)\nprint(\'GT marker tokens removed:\', overall[\'n_gt_markers_removed\'], flush=True)\nprint(\'Không so trực tiếp điểm chính đã lọc với điểm gốc chưa lọc của model khác.\', flush=True)\n\n\n\nprint(\'Save plots\', flush=True)\n\nax = by_category.set_index(\'category\')[[\'mean_cer\', \'mean_wer\']].plot(\n    kind=\'bar\', figsize=(7, 4), rot=0, title=\'OPRB test: mean CER and WER by region\'\n)\nax.set_xlabel(\'Occlusion region\')\nax.set_ylabel(\'Error rate (lower is better)\')\nplt.tight_layout()\nplt.savefig(OUTPUT_DIR / \'cer_wer_by_category.png\', dpi=150)\nplt.close()\n\nax = by_type.set_index(\'occlusion_type\').groupby(level=0)[\'mean_wer\'].mean().sort_values().plot(\n    kind=\'barh\', figsize=(9, 5), title=\'Mean WER by occlusion type (unweighted category average)\'\n)\nax.set_xlabel(\'Mean word error rate (lower is better)\')\nplt.tight_layout()\nplt.savefig(OUTPUT_DIR / \'wer_by_occlusion_type.png\', dpi=150)\nplt.close()\n\n\n\nprint(\'Export results\', flush=True)\n\nrun_config = {\n    **checkpoint_settings,\n    \'categories\': CATEGORIES,\n    \'no_finetuning\': True,\n    \'max_samples_per_category\': MAX_SAMPLES_PER_CATEGORY,\n    \'random_seed\': RANDOM_SEED,\n    \'max_eval_hours_per_run\': MAX_EVAL_HOURS,\n    \'metrics\': overall,\n    \'raw_metrics\': raw_overall,\n    \'ground_truth_storage\': \'Original GT tokens retained in ground_truth; layout tokens removed only while scoring\',\n    \'gpu_name\': DEVICE_NAME if USE_CUDA else None,\n    \'cuda_version\': str(paddle.version.cuda()) if USE_CUDA else None,\n    \'python_version\': platform.python_version(),\n    \'platform\': platform.platform(),\n    \'runtime_python\': sys.executable,\n    \'torch_available_in_runtime\': False,\n    \'runtime_packages_file\': \'runtime_packages.json\',\n    \'jiwer_version\': version(\'jiwer\'),\n    \'pandas_version\': version(\'pandas\'),\n    \'numpy_version\': np.__version__,\n    \'output_token_limit\': None,\n    \'output_token_limit_note\': \'No page-wide VLM token generation limit; crop resolution and CTC decoding remain limitations.\',\n}\n(OUTPUT_DIR / \'run_config.json\').write_text(json.dumps(run_config, indent=2), encoding=\'utf-8\')\nzip_name = shutil.make_archive(str(OUTPUT_DIR), \'zip\', root_dir=OUTPUT_DIR)\nprint(\'Results ZIP:\', zip_name, flush=True)\nprint(\'Tải ZIP trong Output của Kaggle.\', flush=True)\nprint(\'Lần sau: gắn output phiên này làm Input rồi đặt RESUME_FROM đến thư mục CSV/JSON.\', flush=True)\nif not overall[\'evaluation_complete\']:\n    print(\'Đây là kết quả một phần. Tiếp tục checkpoint để hoàn tất tập đã chọn.\', flush=True)\n\n'
BENCHMARK_SCRIPT = OUTPUT_DIR / 'isolated_benchmark.py'
BENCHMARK_SCRIPT.write_text(BENCHMARK_SOURCE, encoding='utf-8')
CONFIG_JSON = OUTPUT_DIR / 'benchmark_settings.json'
CONFIG_JSON.write_text(json.dumps(RUN_SETTINGS, indent=2), encoding='utf-8')
print('Benchmark script:', BENCHMARK_SCRIPT, flush=True)
print('Script SHA-256:', hashlib.sha256(BENCHMARK_SOURCE.encode()).hexdigest(), flush=True)

In [ ]:
import os
import signal
import subprocess

child_env = os.environ.copy()
for key in ('PYTHONPATH', 'PYTHONHOME', 'LD_PRELOAD'):
    child_env.pop(key, None)
child_env['PYTHONNOUSERSITE'] = '1'
# Ưu tiên CUDA libraries trong venv; giữ đường dẫn driver của hệ thống.
library_dirs = sorted(PADDLE_ENV.glob('lib/python*/site-packages/nvidia/*/lib'))
library_dirs += sorted(PADDLE_ENV.glob('lib/python*/site-packages/paddle/libs'))
inherited_paths = [path for path in child_env.get('LD_LIBRARY_PATH', '').split(os.pathsep)
                   if path and 'site-packages' not in path and 'dist-packages' not in path]
child_env['LD_LIBRARY_PATH'] = os.pathsep.join([str(path) for path in library_dirs] + inherited_paths)
LOG_FILE = OUTPUT_DIR / 'execution.log'
command = [str(PADDLE_PYTHON), '-I', '-u', str(BENCHMARK_SCRIPT), '--config', str(CONFIG_JSON)]
print('Bắt đầu benchmark. Theo dõi PROGRESS bên dưới hoặc View Logs của Saved Version.', flush=True)
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, env=child_env, start_new_session=True)
try:
    with LOG_FILE.open('a', encoding='utf-8') as log:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
    returncode = process.wait()
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)
    print('Đã dừng. Checkpoint của các trang ghi xong vẫn còn; chạy lại Mục 3 để tiếp tục.', flush=True)
    raise
finally:
    if process.poll() is None:
        process.terminate()
    if process.stdout is not None:
        process.stdout.close()
if returncode:
    raise RuntimeError(f'Benchmark exited with code {returncode}. Xem traceback phía trên và {LOG_FILE}.')
print('Hoàn tất lượt chạy. Xem kết quả ở Mục 4.', flush=True)

## 4. Xem và tải kết quả

In [ ]:
import csv
import html
import json
from IPython.display import display, HTML, Image, FileLink

config_path = OUTPUT_DIR / 'run_config.json'
if config_path.is_file():
    config = json.loads(config_path.read_text(encoding='utf-8'))
    print('MAIN METRICS (đã lọc nhãn GT):', json.dumps(config['metrics'], indent=2), flush=True)
    print('RAW METRICS (đối chiếu CSV cũ):', json.dumps(config['raw_metrics'], indent=2), flush=True)
    for filename in ['summary_by_category.csv', 'summary_by_occlusion_type.csv']:
        with (OUTPUT_DIR / filename).open(encoding='utf-8', newline='') as stream:
            rows = list(csv.reader(stream))
        table = '<table>' + ''.join('<tr>' + ''.join('<td>' + html.escape(value) + '</td>' for value in row) + '</tr>' for row in rows) + '</table>'
        print(filename, flush=True)
        display(HTML(table))
    for filename in ['cer_wer_by_category.png', 'wer_by_occlusion_type.png', 'preview.png']:
        path = OUTPUT_DIR / filename
        if path.is_file(): display(Image(filename=str(path)))
    archive = OUTPUT_DIR.with_suffix('.zip')
    if archive.is_file(): display(FileLink(str(archive)))
    print('Tải ZIP/thư mục trong Output của Kaggle. Saved Version mới cần RESUME_FROM.', flush=True)
else:
    print('Chưa có run_config.json. Kiểm tra log và checkpoint per_image_results.csv.', flush=True)

## Khi ghi báo cáo

- Đây là một pipeline: detector PP-OCRv5 Mobile và recognizer English PP-OCRv5 Mobile. Ghi cả hai checkpoint, revision/hash, tổng MiB trọng số và phiên bản runtime. Không suy số tham số từ dung lượng file.
- Chỉ số chính dùng Unicode NFKC, khoảng trắng, casefold và bỏ hai token cấu trúc nguyên vẹn `##LTLine##`, `##LTFigure##`. Không bỏ công thức hoặc token khó. GT/prediction gốc vẫn được lưu trong CSV. Chấm lại transcript các model cũ theo cùng chính sách nếu so sánh điểm chính; hoặc đối chiếu đồng nhất bằng `raw_*`.
- Các chỉ số là macro theo ảnh; nhân CER/WER/P/R/F1 với 100 khi dùng %. Token F1 là giao đa tập, không phải F1 detection.
- Thời gian/ảnh đo trong chương trình benchmark: đọc ảnh, tiền xử lý, detector, recognizer, ghép transcript và đồng bộ GPU. Không gồm cài runtime, tải, warm-up, chấm điểm, checkpoint hoặc chuyển log giữa hai tiến trình.
- Dự đoán rỗng vẫn được chấm điểm; lỗi thực thi được lưu riêng và thử lại khi resume. `n_text_regions` đếm vùng PaddleOCR, khác word objects của docTR.
- 10 ảnh chỉ là chạy thử. `evaluation_complete=True` chỉ xác nhận đã xong tập đang chọn; kiểm tra `n_selected` trước khi ghi đủ 9.000 ảnh. Sampling cho bài nộp cần được giảng viên chấp thuận.
- Báo cáo dưới bốn trang A4, giữ template hiện có. So sánh cùng ảnh, chính sách điểm và ghi đúng CPU/GPU. Vẫn cần mô hình thứ năm theo kế hoạch.